In [ ]:
import requests
import time
import pandas as pd
import csv
import os
from datetime import datetime
from dotenv import load_dotenv
load_dotenv()
guardian_api_token = os.getenv('guardian_TOKEN') 
nyt_api_token = os.getenv('nyt_TOKEN')


In [ ]:
def get_content_from_guardian(input,start_date,end_date,api_key):
    content = []
    #is it 50 articels per page, so we go through mutliple pages that match with the same date needed
    page = 1
    while True:
        query = requests.get('https://content.guardianapis.com/search',params={'q' :input,'from-date':start_date,'to-date': end_date,'api-key': api_key,'page-size'  : 50,'page'       : page,
                                                                                  'show-fields': 'headline,body'}
                                                                                  )
        data = query.json()['response']
        content.extend(data['results'])
        if page >= data['pages']:
            break
        page += 1
    return content
#fields is a dict of title and content
guardian_df = pd.DataFrame(get_content_from_guardian('Iran Israel oil energy', '2025-06-01',str(datetime.today().date()),guardian_api_token))
guardian_df = guardian_df[['webPublicationDate','fields']]


In [ ]:
guardian_df.to_csv('guardian_data_since_20250601.csv', index=False, encoding='utf-8')

In [ ]:
def get_content_from_nyt(input,start_date,end_date,api_key):
    content = []
    page = 0
    while True:
        time.sleep(7)  #max 10 queries per min
        query = requests.get('https://api.nytimes.com/svc/search/v2/articlesearch.json',params={'q' :input,'begin_date':start_date,'end_date': end_date,'api-key': api_key,'page': page})
        data = query.json()['response']
        content.extend(data['docs'])
        if len(data['docs']) < 10:  # NYT returns 10 per page
            break
        page += 1
    return (content)

nyt_df = pd.DataFrame(get_content_from_nyt('Iran Israel oil energy', '20250601',(datetime.today().date()).strftime('%Y%m%d'),nyt_api_token))
nyt_df = nyt_df[['pub_date','headline','abstract']]

In [ ]:
nyt_df.to_csv('nyt_data_since_20250601.csv', index=False, encoding='utf-8')